# DGA detection — DSDL char-level neural network

Classifies a domain string as **DGA (1)** or **benign (0)** using a
character-level CNN (Keras). Follows the DSDL container contract: the app
compiles the `# mltkc_*` tagged cells below into a module it calls via
`| fit MLTKContainer algo=dga_neural_network ...` / `| apply ...`.

Train from Splunk with the labeled lookup, then score botsv1 DNS queries.
See `dga/README.md` in the repo for the exact SPL.

In [ ]:
# mltkc_import
# import the libraries DSDL needs available to the compiled module
import json
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

MODEL_DIRECTORY = '/srv/app/model/data/'

# fixed character vocabulary; index 0 is reserved for padding/unknown chars
ALPHABET = 'abcdefghijklmnopqrstuvwxyz0123456789-._'
CHAR_INDEX = {c: i + 1 for i, c in enumerate(ALPHABET)}
MAXLEN = 63


def encode_domains(domains, maxlen=MAXLEN):
    """lowercase domain strings -> fixed-length int sequences."""
    seqs = np.zeros((len(domains), maxlen), dtype=np.int32)
    for r, d in enumerate(domains):
        d = str(d).strip().lower()[:maxlen]
        for c, ch in enumerate(d):
            seqs[r, c] = CHAR_INDEX.get(ch, 0)
    return seqs

In [ ]:
# mltkc_stage
# DEV-ONLY helper: in JupyterLab this re-loads the data + params that DSDL
# staged from your last `... mode=stage` search, so you can iterate on the
# model interactively. Not used when Splunk calls the model in production.
def stage(name):
    with open('data/' + name + '.csv', 'r') as f:
        df = pd.read_csv(f)
    with open('data/' + name + '.json', 'r') as f:
        param = json.load(f)
    return df, param

In [ ]:
# mltkc_init
# build a fresh model. param.options.params may override epochs/batch_size.
def init(df, param):
    opts = param.get('options', {}).get('params', {}) if param else {}
    maxlen = int(opts.get('maxlen', MAXLEN))
    embed_dim = int(opts.get('embed_dim', 32))

    net = keras.Sequential([
        layers.Input(shape=(maxlen,)),
        layers.Embedding(input_dim=len(ALPHABET) + 1, output_dim=embed_dim, mask_zero=True),
        layers.Conv1D(64, 4, activation='relu'),
        layers.GlobalMaxPooling1D(),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid'),
    ])
    net.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    model = {'network': net, 'maxlen': maxlen, 'embed_dim': embed_dim,
             'target': 'is_dga', 'feature': 'domain'}
    return model

In [ ]:
# mltkc_fit
# train on the labeled domains streamed from Splunk.
def fit(model, df, param):
    opts = param.get('options', {}).get('params', {}) if param else {}
    epochs = int(opts.get('epochs', 20))
    batch_size = int(opts.get('batch_size', 32))

    target = model.get('target', 'is_dga')
    feature = model.get('feature', 'domain')
    # DSDL passes target_variables / feature_variables in options too; honor them
    tvars = param.get('options', {}).get('target_variables', []) if param else []
    fvars = param.get('options', {}).get('feature_variables', []) if param else []
    if tvars: target = tvars[0]
    if fvars: feature = fvars[0]
    model['target'], model['feature'] = target, feature

    X = encode_domains(df[feature].values, model['maxlen'])
    y = df[target].astype(int).values
    history = model['network'].fit(X, y, epochs=epochs, batch_size=batch_size,
                                   validation_split=0.2, verbose=2)
    info = {'epochs': epochs, 'batch_size': batch_size,
            'final_loss': float(history.history['loss'][-1]),
            'final_accuracy': float(history.history['accuracy'][-1]),
            'samples': int(len(df))}
    return info

In [ ]:
# mltkc_apply
# score new domains; returns one row per input with prob + 0/1 label.
def apply(model, df, param):
    feature = model.get('feature', 'domain')
    if feature not in df.columns:
        feature = df.columns[0]
    X = encode_domains(df[feature].values, model['maxlen'])
    probs = model['network'].predict(X, verbose=0).reshape(-1)
    out = pd.DataFrame({
        'dga_score': np.round(probs, 4),
        'is_dga_predicted': (probs >= 0.5).astype(int),
    })
    return out

In [ ]:
# mltkc_save
# persist the Keras network + metadata so `apply` can reload it later.
def save(model, name):
    path = MODEL_DIRECTORY + name
    os.makedirs(path, exist_ok=True)
    model['network'].save(os.path.join(path, 'network.keras'))
    meta = {k: model[k] for k in ('maxlen', 'embed_dim', 'target', 'feature')}
    with open(os.path.join(path, 'meta.json'), 'w') as f:
        json.dump(meta, f)
    return model

In [ ]:
# mltkc_load
def load(name):
    path = MODEL_DIRECTORY + name
    with open(os.path.join(path, 'meta.json'), 'r') as f:
        meta = json.load(f)
    net = keras.models.load_model(os.path.join(path, 'network.keras'))
    model = dict(meta)
    model['network'] = net
    return model

In [ ]:
# mltkc_summary
def summary(model=None):
    s = {'version': {'tensorflow': tf.__version__},
         'algorithm': 'dga_neural_network (char-level CNN)'}
    if model and 'network' in model:
        s['maxlen'] = model.get('maxlen')
        s['params'] = int(model['network'].count_params())
    return s

## Local sanity check (optional, run inside JupyterLab)

After you've trained once from Splunk with `mode=stage`, you can iterate here:

In [ ]:
# DEV-ONLY: train + eval on the staged data without going through Splunk
# df, param = stage('dga_neural_network')
# model = init(df, param)
# print(fit(model, df, param))
# print(apply(model, df.head(10), param))
# print(summary(model))